# GCN Example 2 (PyTorch) — Graph Classification (synthetic graphs)

PyTorch version of `graph_classification.ipynb`. We classify whole graphs (not nodes)
into **sparse** (class 0) vs **dense** (class 1) Erdős–Rényi random graphs.

The pipeline is `node features → GCN layers → readout (pooling) → MLP → graph label`. The
readout collapses per-node embeddings into one vector per graph; here we concatenate mean
and max pooling. Batched message passing uses `torch.bmm` (batched matrix multiply).

> Self-contained and fast on CPU. Needs `torch`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Generate the synthetic dataset

Each graph: `N = 20` nodes, edges sampled with probability `p` (`0.15` for sparse, `0.40`
for dense). Node feature = degree (scaled). We normalize each adjacency (self-loops +
symmetric normalization) and stack into batched tensors.

In [ ]:
N = 20
N_PER_CLASS = 250

def random_graph(p):
    upper = (np.random.rand(N, N) < p).astype('float32')
    upper = np.triu(upper, k=1)
    return upper + upper.T

def normalize(A):
    A_tilde = A + np.eye(N, dtype='float32')
    d = A_tilde.sum(1)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d))
    return (d_inv_sqrt @ A_tilde @ d_inv_sqrt).astype('float32')

X_list, Ahat_list, y_list = [], [], []
for label, p in [(0, 0.15), (1, 0.40)]:
    for _ in range(N_PER_CLASS):
        A = random_graph(p)
        deg = A.sum(1, keepdims=True).astype('float32')
        X_list.append(deg / N)
        Ahat_list.append(normalize(A))
        y_list.append(label)

X = torch.tensor(np.stack(X_list))
Ahat = torch.tensor(np.stack(Ahat_list))
y = torch.tensor(np.array(y_list), dtype=torch.long)

perm = torch.randperm(len(y))
X, Ahat, y = X[perm], Ahat[perm], y[perm]
split = int(0.8 * len(y))
Xtr, Xte = X[:split].to(device), X[split:].to(device)
Atr, Ate = Ahat[:split].to(device), Ahat[split:].to(device)
ytr, yte = y[:split].to(device), y[split:].to(device)
print('Dataset:', tuple(X.shape), '| train:', len(ytr), '| test:', len(yte))

## 2. The model

`nn.Linear` applied to `(B, N, F)` transforms the last axis (the `X W` step);
`torch.bmm(A_hat, ...)` mixes neighbours. Two GCN layers, then mean+max readout, then a
small classifier.

In [ ]:
class GraphGCN(nn.Module):
    def __init__(self, in_features=1, hidden=32, classes=2):
        super().__init__()
        self.lin1 = nn.Linear(in_features, hidden, bias=False)
        self.lin2 = nn.Linear(hidden, hidden, bias=False)
        self.classifier = nn.Sequential(
            nn.Linear(2 * hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, classes),
        )

    def forward(self, X, A_hat):
        h = F.relu(torch.bmm(A_hat, self.lin1(X)))     # GCN layer 1
        h = F.relu(torch.bmm(A_hat, self.lin2(h)))     # GCN layer 2
        graph_emb = torch.cat([h.mean(dim=1), h.amax(dim=1)], dim=1)  # readout
        return self.classifier(graph_emb)

model = GraphGCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
print(model)

## 3. Train (full-batch) and evaluate

In [ ]:
EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    model.train()
    logits = model(Xtr, Atr)
    loss = F.cross_entropy(logits, ytr)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            train_acc = (logits.argmax(1) == ytr).float().mean().item()
            test_acc = (model(Xte, Ate).argmax(1) == yte).float().mean().item()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={loss.item():.4f}  '
              f'train_acc={train_acc:.3f}  test_acc={test_acc:.3f}')

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(Xte, Ate).argmax(1).cpu().numpy()
yte_np = yte.cpu().numpy()
print(f'Test accuracy: {(pred == yte_np).mean():.1%}\n')
cm = np.zeros((2, 2), dtype=int)
for t, p in zip(yte_np, pred):
    cm[t, p] += 1
print('Confusion matrix (rows=true, cols=pred):')
print('              pred sparse  pred dense')
print(f'true sparse   {cm[0,0]:10d}  {cm[0,1]:10d}')
print(f'true dense    {cm[1,0]:10d}  {cm[1,1]:10d}')

### Things to try

- Move the probabilities closer (`0.25` vs `0.30`) to make it harder.
- Try sum pooling instead of mean+max.
- Use a structure-based contrast (stochastic block model vs uniform random with equal
  edge count) so density alone can't solve it.
- Support variable-size graphs with block-diagonal batching.